# Obtenção de Dados — Pesquisa

Notebook de **obtenção de dados** (sem treino, sem inferência de modelo): cada célula roda um script de `src/pesquisa/`, que mede alguma característica do corpus real do Manga109 e gera CSV + gráfico prontos pra apresentação da pesquisa.

**Checklist antes de rodar:**
1. Anexe o dataset de **anotações** do Manga109 no painel lateral direito em **+ Add Input → Datasets** (as imagens não são necessárias pra esses scripts -- só a transcrição/bbox de linha do XML).
2. GPU não é necessária (scripts aqui são só contagem de texto, não rodam modelo).
3. **Run All**.


## 1. Instalar dependências

In [ ]:
!pip install -q tqdm matplotlib
print("Dependencias instaladas.")


## 2. Configurar repositório

Clona (ou atualiza) o repositório em `/kaggle/working/`.

In [ ]:
import os
import sys
import shutil

WORK_DIR  = "/kaggle/working"
REPO_NAME = "Detector-de-kanjis-n1"
REPO_DIR  = os.path.join(WORK_DIR, REPO_NAME)
REPO_URL  = f"https://github.com/MiguelMussalam/{REPO_NAME}.git"

is_valid_repo = os.path.isdir(os.path.join(REPO_DIR, ".git"))

if not is_valid_repo:
    if os.path.exists(REPO_DIR):
        print(f"Diretorio {REPO_DIR} existe mas nao e um repo git valido. Removendo...")
        shutil.rmtree(REPO_DIR)
    print(f"Clonando {REPO_URL} ...")
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo valido encontrado. Atualizando...")
    !git -C {REPO_DIR} pull

assert os.path.isfile(os.path.join(REPO_DIR, "config.py")), \
    f"config.py nao encontrado em {REPO_DIR} -- verifique se o clone funcionou"

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f"Diretorio de trabalho: {os.getcwd()}")


## 3. Construir ground truth full-corpus

Percorre todo XML do Manga109 (bbox de linha `<text>` + transcrição oficial) -- rápido (segundos), não depende de nenhum modelo. Ver `src/helper/manga109_corpus.py`.

In [ ]:
!python -m src.helper.manga109_corpus


## 4. `kanji_coverage` — cobertura de N1 e comparação N1-N5

Duas medidas a partir da frequência real de texto no corpus (não depende de detecção/alinhamento):

1. **Cobertura N1**: quantas vezes cada um dos 1232 kanji N1 (nosso alvo de classificação) aparece de verdade no corpus -- quantas classes nunca aparecem, quantas são raras.
2. **Comparação N1-N5**: frequência média de aparição por kanji em cada nível JLPT -- mostra quantitativamente que N1 é mais raro no texto real do que N5, como esperado pelo design do próprio JLPT.

Ver `src/pesquisa/kanji_coverage.py`.

In [ ]:
from IPython.display import Image, display

from src.pesquisa.kanji_coverage import carregar_corpus, cobertura_n1, comparar_niveis

gt = carregar_corpus()
print(f"Corpus carregado: {len(gt['paginas'])} paginas")

res_n1 = cobertura_n1(gt)
display(Image(filename=res_n1["png"]))

res_niveis = comparar_niveis(gt)
display(Image(filename=res_niveis["png"]))


## 5. Compactar e baixar

In [ ]:
import zipfile
from IPython.display import FileLink, display

zip_name = "/kaggle/working/dados_pesquisa.zip"
pesquisa_dir = os.path.join(REPO_DIR, "data", "pesquisa")

with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(pesquisa_dir):
        for fname in files:
            fpath = os.path.join(root, fname)
            zipf.write(fpath, os.path.relpath(fpath, pesquisa_dir))

print(f"Zip criado: {zip_name}")
display(FileLink(zip_name))
